# British Airways: Predicting Customer Buying Behaviour

**Goal:** predict whether a customer completes a booking (`booking_complete`) and identify which variables drive that outcome.

**Workflow:** 1) Load and explore, 2) Clean, 3) Engineer features, 4) Encode, 5) Train a Random Forest, 6) Cross-validate and evaluate, 7) Interpret feature importance, 8) Summarise for the slide.

Place `customer_booking.csv` in the same folder as this notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, cross_val_score, cross_val_predict)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, precision_score, recall_score, f1_score)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

## 1. Load the data

In [ ]:
# ISO-8859-1 avoids the UnicodeDecodeError that this file usually triggers
df = pd.read_csv("customer_booking.csv", encoding="ISO-8859-1")
print("Shape:", df.shape)
df.head()

## 2. Explore the data

In [ ]:
df.info()
df.describe().T

In [ ]:
print("Missing values per column:\n", df.isna().sum(), "\n")
print("Duplicate rows:", df.duplicated().sum())
print("\nUnique values:")
print(df.nunique().sort_values(ascending=False))

In [ ]:
# Target balance: expect heavy imbalance (~15% bookings)
target_share = df["booking_complete"].value_counts(normalize=True).rename("share")
print(target_share.round(3))
print("\nBaseline accuracy if we always predict 'no booking':", round(target_share.max(), 3))

In [ ]:
# Distributions of the key numeric variables (look for extreme values)
num_cols = ["num_passengers", "purchase_lead", "length_of_stay", "flight_hour", "flight_duration"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, c in zip(axes.ravel(), num_cols):
    sns.histplot(df[c], bins=40, ax=ax)
    ax.set_title(c)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

print(df[["purchase_lead", "length_of_stay"]].quantile([.5, .9, .95, .99, 1.0]))

In [ ]:
# Booking rate by category
cat_cols = ["sales_channel", "trip_type", "flight_day",
            "wants_extra_baggage", "wants_preferred_seat", "wants_in_flight_meals"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, c in zip(axes.ravel(), cat_cols):
    rate = df.groupby(c)["booking_complete"].mean().sort_values(ascending=False)
    sns.barplot(x=rate.index.astype(str), y=rate.values, ax=ax)
    ax.set_title(f"Booking rate by {c}")
    ax.set_ylabel("share completing booking")
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Booking rate by top 10 booking origins and by lead-time band
top_orig = df["booking_origin"].value_counts().head(10).index
print(df[df["booking_origin"].isin(top_orig)]
      .groupby("booking_origin")["booking_complete"].agg(["mean", "count"])
      .sort_values("mean", ascending=False).round(3))

lead_band = pd.cut(df["purchase_lead"], [-1, 7, 30, 90, 180, 1000])
print("\n", df.groupby(lead_band)["booking_complete"].agg(["mean", "count"]).round(3))

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.select_dtypes("number").corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix (numeric columns)")
plt.show()

## 3. Clean the data

Decisions made here (mention these on your slide or in interviews):
- Drop exact duplicate rows.
- Map `flight_day` from text to 1-7.
- Cap `purchase_lead` and `length_of_stay` at the 99th percentile. Random Forests tolerate outliers, but capping keeps plots and engineered ratios sensible.

In [ ]:
d = df.drop_duplicates().reset_index(drop=True).copy()
print("Rows after dropping duplicates:", len(d))

day_map = {"Mon": 1, "Tue": 2, "Wed": 3, "Thu": 4, "Fri": 5, "Sat": 6, "Sun": 7}
d["flight_day"] = d["flight_day"].map(day_map)
assert d["flight_day"].notna().all(), "Unmapped flight_day values found"

for c in ["purchase_lead", "length_of_stay"]:
    cap = d[c].quantile(0.99)
    d[c] = d[c].clip(upper=cap)
    print(f"{c}: capped at {cap:.0f}")

## 4. Feature engineering

New features and the intuition behind them:

| Feature | Idea |
|---|---|
| `total_extras` | Customers who add baggage, seats or meals are more engaged |
| `lead_per_passenger` | Planning time relative to group size |
| `lead_bucket` | Last-minute (0), short (1), medium (2), long (3) planners |
| `is_last_minute` | Booking within 7 days of departure |
| `is_long_stay` | Stay over 14 days |
| `is_weekend` | Departure on Sat/Sun |
| `time_of_day` | Night / morning / afternoon / evening departure |
| `is_long_haul` | Flight duration of 7+ hours |
| `route_origin`, `route_dest` | Airport codes parsed from the route |

In [ ]:
extras = ["wants_extra_baggage", "wants_preferred_seat", "wants_in_flight_meals"]

d["total_extras"]        = d[extras].sum(axis=1)
d["lead_per_passenger"]  = d["purchase_lead"] / d["num_passengers"]
d["lead_bucket"]         = pd.cut(d["purchase_lead"], [-1, 7, 30, 90, 10_000], labels=[0, 1, 2, 3]).astype(int)
d["is_last_minute"]      = (d["purchase_lead"] <= 7).astype(int)
d["is_long_stay"]        = (d["length_of_stay"] > 14).astype(int)
d["is_weekend"]          = d["flight_day"].isin([6, 7]).astype(int)
d["time_of_day"]         = pd.cut(d["flight_hour"], [-1, 5, 11, 17, 23],
                                  labels=[0, 1, 2, 3]).astype(int)   # 0 night, 1 morning, 2 afternoon, 3 evening
d["is_long_haul"]        = (d["flight_duration"] >= 7).astype(int)
d["route_origin"]        = d["route"].str[:3]
d["route_dest"]          = d["route"].str[3:]

engineered_cols = ["total_extras", "lead_per_passenger", "lead_bucket", "is_last_minute",
                   "is_long_stay", "is_weekend", "time_of_day", "is_long_haul"]
d[engineered_cols].describe().T

## 5. Encode categorical variables

- `sales_channel`, `trip_type`: one-hot encoding.
- `booking_origin` (~100 countries): keep the 15 most common as one-hot columns and group the rest as "Other".
- `route`, `route_origin`, `route_dest` (hundreds of values): **frequency encoding**, meaning each value is replaced by how common it is. This uses no target information, so there is no leakage, and it avoids hundreds of sparse columns.

In [ ]:
# Frequency encoding for high-cardinality columns
for c in ["route", "route_origin", "route_dest"]:
    d[f"{c}_freq"] = d[c].map(d[c].value_counts(normalize=True))

# Group rare booking origins
top15 = d["booking_origin"].value_counts().head(15).index
d["booking_origin_grp"] = d["booking_origin"].where(d["booking_origin"].isin(top15), "Other")

# One-hot encoding
d = pd.get_dummies(d, columns=["sales_channel", "trip_type", "booking_origin_grp"],
                   prefix=["channel", "trip", "origin"], dtype=int)

# Drop raw high-cardinality text columns
model_df = d.drop(columns=["route", "route_origin", "route_dest", "booking_origin"])

X = model_df.drop(columns="booking_complete")
y = model_df["booking_complete"]
print("Feature matrix:", X.shape)
print("Features:", list(X.columns))

## 6. Train / test split

Hold out 20% of the data for a final check. The split is **stratified** so both sets keep the ~15% booking rate.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, "| booking rate:", round(y_train.mean(), 3))
print("Test: ", X_test.shape,  "| booking rate:", round(y_test.mean(), 3))

## 7. Does feature engineering help?

Compare cross-validated ROC-AUC with and without the engineered features. This is useful evidence for the write-up.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def make_rf(n=300):
    return RandomForestClassifier(
        n_estimators=n, min_samples_leaf=5, max_features="sqrt",
        class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1)

base_cols = [c for c in X_train.columns if c not in engineered_cols]

auc_base = cross_val_score(make_rf(200), X_train[base_cols], y_train, cv=cv, scoring="roc_auc")
auc_full = cross_val_score(make_rf(200), X_train,            y_train, cv=cv, scoring="roc_auc")

print(f"Original features only : AUC = {auc_base.mean():.3f} +/- {auc_base.std():.3f}")
print(f"With engineered features: AUC = {auc_full.mean():.3f} +/- {auc_full.std():.3f}")

## 8. Train the Random Forest and cross-validate

Cross-validation on the training set gives an honest estimate of performance and its variability.

In [ ]:
rf = make_rf(300)

scoring = {"roc_auc": "roc_auc", "avg_precision": "average_precision",
           "precision": "precision", "recall": "recall", "f1": "f1", "accuracy": "accuracy"}
cv_res = cross_validate(rf, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)

cv_table = pd.DataFrame({m: [cv_res[f"test_{m}"].mean(), cv_res[f"test_{m}"].std()] for m in scoring},
                        index=["mean", "std"]).T.round(3)
print("5-fold cross-validation (default 0.5 threshold):")
cv_table

In [ ]:
rf.fit(X_train, y_train)
proba_test = rf.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)

print("Test ROC-AUC:", round(roc_auc_score(y_test, proba_test), 3))
print("Test average precision (PR-AUC):", round(average_precision_score(y_test, proba_test), 3))
print("\n", classification_report(y_test, pred_test, digits=3))

## 9. Threshold tuning

At 0.5 a balanced Random Forest often has low precision or recall for the booking class. We pick the threshold that maximises F1 using **out-of-fold predictions on the training set only**, then check it once on the test set. This avoids tuning on the test data.

In [ ]:
oof = cross_val_predict(make_rf(300), X_train, y_train, cv=cv, method="predict_proba")[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof)
f1s = 2 * prec * rec / (prec + rec + 1e-12)
best_thr = thr[np.argmax(f1s[:-1])]
print(f"Best threshold from CV (max F1): {best_thr:.3f}")

pred_tuned = (proba_test >= best_thr).astype(int)
summary = pd.DataFrame({
    "threshold": [0.5, best_thr],
    "precision": [precision_score(y_test, pred_test), precision_score(y_test, pred_tuned)],
    "recall":    [recall_score(y_test, pred_test),    recall_score(y_test, pred_tuned)],
    "f1":        [f1_score(y_test, pred_test),        f1_score(y_test, pred_tuned)],
}, index=["default 0.5", "tuned"]).round(3)
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

# ROC curve
fpr, tpr, _ = roc_curve(y_test, proba_test)
axes[0].plot(fpr, tpr, label=f"Random Forest (AUC = {roc_auc_score(y_test, proba_test):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", label="Random guess")
axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curve (test set)")
axes[0].legend()

# Precision-recall curve
p, r, _ = precision_recall_curve(y_test, proba_test)
axes[1].plot(r, p)
axes[1].axhline(y_test.mean(), color="k", ls="--", label="Baseline (booking rate)")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall curve (test set)")
axes[1].legend()

# Confusion matrix at tuned threshold
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_tuned),
                       display_labels=["No booking", "Booking"]).plot(ax=axes[2], colorbar=False)
axes[2].set_title(f"Confusion matrix (threshold {best_thr:.2f})")

plt.tight_layout()
plt.savefig("model_evaluation.png", dpi=200)
plt.show()

## 10. Feature importance

Two views:
- **Impurity-based importance** (built into Random Forest): fast, but biased towards high-cardinality and continuous features.
- **Permutation importance** (on the test set): measures the drop in ROC-AUC when a feature is shuffled. More reliable, so use this one for the slide.

In [ ]:
imp_impurity = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

perm = permutation_importance(rf, X_test, y_test, scoring="roc_auc",
                              n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
imp_perm = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)

top_n = 12
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
imp_impurity.head(top_n)[::-1].plot(kind="barh", ax=axes[0], color="#1f4e79")
axes[0].set_title("Impurity-based importance")
imp_perm.head(top_n)[::-1].plot(kind="barh", ax=axes[1], color="#c8102e")
axes[1].set_title("Permutation importance (drop in ROC-AUC)")
plt.tight_layout()
plt.show()

pd.DataFrame({"impurity": imp_impurity, "permutation": imp_perm}).sort_values("permutation", ascending=False).head(15).round(4)

In [ ]:
# Clean single chart for the slide
plot_imp = imp_perm.head(10)[::-1]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(plot_imp.index, plot_imp.values, color="#1f4e79")
ax.set_xlabel("Drop in ROC-AUC when feature is shuffled")
ax.set_title("What drives a customer completing a booking?", loc="left", fontweight="bold")
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=250, transparent=False)
plt.show()

## 11. Direction of the effect

Importance says *what* matters, not *how*. Booking rate across the range of the top drivers gives the direction and gives you slide-ready insights.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

lead = pd.cut(d["purchase_lead"], [-1, 7, 30, 90, 180, 1000])
d.groupby(lead)["booking_complete"].mean().plot(kind="bar", ax=axes[0], color="#1f4e79")
axes[0].set_title("Booking rate by purchase lead (days)")

d.groupby("total_extras")["booking_complete"].mean().plot(kind="bar", ax=axes[1], color="#1f4e79")
axes[1].set_title("Booking rate by number of add-ons")

stay = pd.cut(d["length_of_stay"], [-1, 3, 7, 14, 30, 1000])
d.groupby(stay)["booking_complete"].mean().plot(kind="bar", ax=axes[2], color="#1f4e79")
axes[2].set_title("Booking rate by length of stay (days)")

for ax in axes:
    ax.set_ylabel("booking rate"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 12. Summary numbers for the slide

In [ ]:
print("=== NUMBERS FOR THE SLIDE ===")
print(f"Rows used                    : {len(model_df):,}")
print(f"Booking rate                 : {y.mean():.1%}")
print(f"CV ROC-AUC (5-fold)          : {cv_res['test_roc_auc'].mean():.3f} +/- {cv_res['test_roc_auc'].std():.3f}")
print(f"Test ROC-AUC                 : {roc_auc_score(y_test, proba_test):.3f}")
print(f"Test precision / recall (tuned): {precision_score(y_test, pred_tuned):.2f} / {recall_score(y_test, pred_tuned):.2f}")
print(f"Test accuracy @0.5           : {(pred_test == y_test).mean():.1%}  (baseline {1 - y.mean():.1%})")
print("\nTop 5 drivers (permutation):")
print(imp_perm.head(5).round(4).to_string())

## Notes and limitations
- Importance shows association, not causation.
- The target is imbalanced, so ROC-AUC and precision/recall are reported alongside accuracy.
- Next steps: try gradient boosting (e.g. LightGBM), tune hyperparameters with `RandomizedSearchCV`, and test target encoding for `route` and `booking_origin` inside a CV pipeline.